In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

from pathlib import Path

In [20]:

# Define preprocessing transforms
# MobileNet expects 224x224 RGB images normalized with ImageNet stats
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load datasets
train_dataset = datasets.ImageFolder('VisDrone-objects/train', transform=transform)
val_dataset = datasets.ImageFolder('VisDrone-objects/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(train_dataset.classes)
print(f"Number of classes: {num_classes}")

Number of classes: 10


In [21]:
# Load pre-trained MobileNetV2
model = models.mobilenet_v2(pretrained=True)

# Option 1: Replace the last classifier layer
num_classes = len(train_dataset.classes)
model.classifier[-1] = nn.Linear(in_features=1280, out_features=num_classes)

# Option 2: Replace entire classifier (more flexible)
# in_features = model.classifier[0].in_features
# model.classifier = nn.Sequential(
#     nn.Linear(in_features, 512),
#     nn.ReLU(),
#     nn.Dropout(0.2),
#     nn.Linear(512, num_classes)
# )

/home/gpu_03/class-agnostic/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/gpu_03/class-agnostic/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [22]:
# Freeze all feature extraction layers
for param in model.features.parameters():
    param.requires_grad = False

# Verify only classifier parameters are trainable
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

Trainable parameters: 12810


In [23]:
criterion = nn.CrossEntropyLoss()

# Only optimize parameters that require gradients (the new classifier)
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001,
    momentum=0.9
)

# Alternatively, use Adam
# optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

In [25]:

log_dir = Path('mobilenet_logs')
log_dir.mkdir(exist_ok=True)

In [ ]:
num_epochs =50
best_val_acc = 0.0

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100. * correct / total
    train_loss = running_loss / len(train_loader)
    
    # Validation phase
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * val_correct / val_total
    
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Acc: {val_acc:.2f}%')
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{log_dir}/best_mobilenet_model.pth')

Epoch [1/50]
Train Loss: 1.2186, Train Acc: 59.22%
Val Acc: 56.47%
Epoch [2/50]
Train Loss: 1.2168, Train Acc: 59.35%
Val Acc: 55.17%
Epoch [3/50]
Train Loss: 1.2157, Train Acc: 59.32%
Val Acc: 56.70%
Epoch [4/50]
Train Loss: 1.2152, Train Acc: 59.36%
Val Acc: 56.70%
Epoch [5/50]
Train Loss: 1.2129, Train Acc: 59.26%
Val Acc: 58.06%
Epoch [6/50]
Train Loss: 1.2154, Train Acc: 59.31%
Val Acc: 57.34%
Epoch [7/50]
Train Loss: 1.2125, Train Acc: 59.36%
Val Acc: 57.37%
Epoch [8/50]
Train Loss: 1.2129, Train Acc: 59.33%
Val Acc: 56.96%
Epoch [9/50]
Train Loss: 1.2138, Train Acc: 59.31%
Val Acc: 57.01%
Epoch [10/50]
Train Loss: 1.2128, Train Acc: 59.36%
Val Acc: 56.75%
Epoch [11/50]
Train Loss: 1.2135, Train Acc: 59.40%
Val Acc: 57.64%
Epoch [12/50]
Train Loss: 1.2127, Train Acc: 59.26%
Val Acc: 57.93%
Epoch [13/50]
Train Loss: 1.2125, Train Acc: 59.40%
Val Acc: 56.35%
Epoch [14/50]
Train Loss: 1.2135, Train Acc: 59.30%
Val Acc: 56.52%
Epoch [15/50]
Train Loss: 1.2139, Train Acc: 59.29%
Val A